# DFedSET 超参数分析

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
from utils import ResultLoader
from matplotlib.colors import ListedColormap, BoundaryNorm

# 设置绘图风格
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'figure.titlesize': 18
})

# 初始化加载器，指向 Ray 实验结果目录
loader = ResultLoader("../results_ray")

data_cache = {}
def cached_load(algo, **kwargs):
    key = (algo, tuple(sorted(kwargs.items())))
    if key not in data_cache:
        data_cache[key] = loader.load(algo, **kwargs)
    return data_cache[key]

In [ ]:
# 实验配置
common_args = {
    "dataset": "cifar10",
    "partition": "dirichlet",
    "num_clients": 10,
    "alpha": 0.1,
    "epochs": 1,
    "batch_size": 64,
    "lr": 0.01,
    "adj_type": "ring",
}
algorithms_comm = {
    "Local":     {"algo": "local",   "kwargs": {}},
    "FedAvg":    {"algo": "fedavg",   "kwargs": {}},
    "EF-HC":     {"algo": "efhc",   "kwargs": {"event_r": 250, "bandwidth_mean": 5000, "bandwidth_std": 0.0}},
    "DFedAvgM":  {"algo": "dfedavgm", "kwargs": {}},
    "DisPFL":    {"algo": "dispfl",  "kwargs": {"dense_ratio": 0.5, "anneal_factor": 1.0}},
    "L2C":       {"algo": "l2c",     "kwargs": {"val_ratio": 0.1, "lr_alpha": 0.1, "prune_round": 20, "prune_num": 0}},
    "PearFL":    {"algo": "pearfl",  "kwargs": {"lamda": 0.1}},
    "DFedPGP":   {"algo": "dfedpgp", "kwargs": {"local_v_epochs": 1, "lr_v": 0.01, "momentum_v": 0.0, "weight_decay_v": 0.0}},
    "ProxyFL":   {"algo": "proxyfl", "kwargs": {"mu": 0.01}},
    "FedProto":  {"algo": "fedproto","kwargs": {"mu": 0.1}},
    "DFedSET":   {"algo": "dfedset", "kwargs": {"lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}},
}
plot_fontsize = 24
plot_linewidth = 2

def calc_cum_comm(data, algo, num_clients, join_ratio):
    """返回累计通信量（累计参数传输数量）"""
    acc_dict = data.get("acc", {})
    if isinstance(acc_dict, dict) and "model" in acc_dict:
        n_rounds = len(acc_dict["model"])
    else:
        n_rounds = len(acc_dict) if isinstance(acc_dict, list) else 0

    if algo == "dfedset":
        num_tr = data.get("num_triggered", [])
        per_round = np.array(num_tr[:n_rounds]) * BODY_PARAMS + num_clients * DFEDSET_EXTRA
    elif algo == "efhc":
        num_tr = data.get("num_triggered", [])
        per_round = np.array(num_tr[:n_rounds]) * MODEL_PARAMS
    elif algo == "local":
        per_round = np.zeros(n_rounds)
    elif algo == "l2c":
        per_round = np.full(n_rounds, num_clients * 2 * MODEL_PARAMS)
    elif algo == "dispfl":
        per_round = np.full(n_rounds, int(num_clients * MODEL_PARAMS * 0.5))
    elif algo == "dfedpgp":
        per_round = np.full(n_rounds, int(num_clients * BODY_PARAMS))
    elif algo == "fedproto":
        per_round = np.full(n_rounds, int(num_clients * PROTOS_PARAMS))
    elif algo == "pearfl":
        per_round = np.full(n_rounds, int(num_clients * (MODEL_PARAMS + PROTOS_PARAMS)))
    else:
        per_round = np.full(n_rounds, int(num_clients * join_ratio * MODEL_PARAMS))
    return np.cumsum(per_round)

In [ ]:
# ====== 可配置参数 ======
ABLATE_NAME = None   # 可选：None, "relay", "aggregator", "confidence_count", "confidence_none", "trigger_global_gamma_global_0.01", "trigger_global_gamma_global_0.05", "trigger_all"
EPOCHS = 1
NUM_CLIENTS = 20
LAMBDA_SA = 10.0
ETA = 0.9
LAMBDA_SO = 0.1
# ========================

args = {**common_args, "epochs": EPOCHS, "num_clients": NUM_CLIENTS,
        "lambda_sa": LAMBDA_SA, "eta": ETA, "lambda_so": LAMBDA_SO}
data = cached_load("dfedset", ablate_name=ABLATE_NAME, **args, specific_run=0)
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到")

acc = data["acc"]["model"]

# Build title
if ABLATE_NAME is None:
    title = f"DFedSET (λ_sa={LAMBDA_SA}, λ_so={LAMBDA_SO})"
elif ABLATE_NAME == "relay":
    title = "DFedSET - No Relay"
elif ABLATE_NAME == "aggregator":
    title = "DFedSET - Plain Aggr."
elif ABLATE_NAME == "confidence_count":
    title = "DFedSET - Conf=count"
elif ABLATE_NAME == "confidence_none":
    title = "DFedSET - Conf=none"
elif ABLATE_NAME == "trigger_global_gamma_global_0.01":
    title = "DFedSET - Global gamma=0.01"
elif ABLATE_NAME == "trigger_global_gamma_global_0.05":
    title = "DFedSET - Global gamma=0.05"
elif ABLATE_NAME == "trigger_all":
    title = "DFedSET - All Trigger"
else:
    title = f"DFedSET - {ABLATE_NAME}"

plt.figure(figsize=(10, 7))
plt.plot(acc, linewidth=plot_linewidth, color="#D62728")
plt.xlabel("Round", fontsize=plot_fontsize)
plt.ylabel("Accuracy (%)", fontsize=plot_fontsize)
plt.tick_params(labelsize=plot_fontsize)
plt.title(title, fontsize=plot_fontsize)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/dfedset_acc_curve.pdf", bbox_inches="tight")
plt.show()

## Fig 1: 通信量 VS Acc，多算法对比

**所需实验：** 每种算法至少一个完整 run（1000 rounds），epochs 对齐。
所有算法同模型（cnn），同 join_ratio（1.0）。

- dfedset: epochs=1, λ_sa=10.0, λ_so=0.1
- efhc: epochs=1, event_r=250, bandwidth_mean=5000
- l2c, dispfl, pearfl, dfedavgm, dfedpgp, proxyfl, local, fedproto: epochs=1

In [ ]:
MODEL_PARAMS = 2122186     # 完整 CNN 参数量
BODY_PARAMS = 2117056      # DFedPGP 提取器（不含分类头）
PROTOS_PARAMS = 5120       # 原型 [10, 512]
DFEDSET_EXTRA = 5130       # S[10,512] + W[10]
PARAM_TO_GB = 4 / (1024 * 1024 * 1024)  # 每个参数 4 bytes → GB

# 选择需要绘制的算法
plot_algorithms = [
    "DFedSET",
    "EF-HC",
    "DFedAvgM",
    # "DisPFL",
    # "L2C",
    "PearFL",
    "DFedPGP",
    # "ProxyFL",
]
# 自定义各算法曲线的颜色和线型
line_styles = {
    "DFedSET":   {"color": "#D62728", "linestyle": "-"},
    "EF-HC":     {"color": "#1F77B4", "linestyle": "-"},
    "DFedAvgM":  {"color": "#9467BD", "linestyle": "-"},
    "DFedPGP":   {"color": "#FF7F0E", "linestyle": "-"},
    "PearFL":    {"color": "#2CA02C", "linestyle": "-"},
    "FedAvg":    {"color": "black", "linestyle": "-."},
    "Local":     {"color": "black", "linestyle": "--"},
    "ProxyFL":   {"color": "#8C564B", "linestyle": "-"},
    "L2C":       {"color": "#E377C2", "linestyle": "-"},
}

plt.figure(figsize=(10, 8))
for label, spec in algorithms_comm.items():
    if plot_algorithms and label not in plot_algorithms:
        continue
    args = {**common_args, **spec["kwargs"]}
    data = cached_load(spec["algo"], **args, specific_run=0)
    if data is None:
        print(f"  [跳过] {label}: 无结果")
        continue

    acc_dict = data.get("acc", {})
    if isinstance(acc_dict, dict) and "model" in acc_dict:
        acc = acc_dict["model"]
    elif isinstance(acc_dict, list):
        acc = acc_dict
    else:
        continue

    cum_comm = calc_cum_comm(data, spec["algo"], 10, 1.0) * PARAM_TO_GB
    min_len = min(len(cum_comm), len(acc))
    style = line_styles.get(label, {"color": None, "linestyle": "-"})
    total_gb = cum_comm[min_len-1]
    plt.plot(cum_comm[:min_len], acc[:min_len], label=f"{label} ({total_gb:.2f} GB)", linewidth=plot_linewidth, color=style["color"], linestyle=style["linestyle"])

# 绘制 FedAvg 基准线 (使用 epochs=10)
fedavg_data = cached_load("fedavg", **common_args, specific_run=0)
if fedavg_data and "acc" in fedavg_data:
    fedavg_acc = max(fedavg_data["acc"])
    fedavg_total_comm = calc_cum_comm(fedavg_data, "fedavg", 10, 1.0) * PARAM_TO_GB
    style = line_styles.get("FedAvg", {"color": "gray", "linestyle": "--"})
    plt.axhline(y=fedavg_acc, color=style["color"], linestyle=style["linestyle"], linewidth=plot_linewidth, label=f"FedAvg ({fedavg_total_comm[-1]:.2f} GB)")

local_data = cached_load("local", **common_args, specific_run=0)
if local_data and "acc" in local_data:
    local_acc = max(local_data["acc"])
    local_total_comm = calc_cum_comm(local_data, "local", 10, 1.0) * PARAM_TO_GB
    style = line_styles.get("Local", {"color": "black", "linestyle": "--"})
    plt.axhline(y=local_acc, color=style["color"], linestyle=style["linestyle"], linewidth=plot_linewidth, label=f"Local ({local_total_comm[-1]:.2f} GB)")

plt.xlabel("Cumulative Communication (GB)", fontsize=plot_fontsize)
plt.ylabel("Accuracy (%)", fontsize=plot_fontsize)
plt.xlim(0, 40)
plt.ylim(30, 92)
plt.tick_params(labelsize=plot_fontsize)
plt.legend(fontsize=plot_fontsize-2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/comm_vs_acc.pdf", bbox_inches="tight")
plt.show()

## Fig 2: 触发策略对比

**所需实验：** DFedSET，固定 λ_sa=10.0, λ_so=0.1, epochs=1, cifar10, dirichlet, alpha=0.1
四种触发策略: Adaptive (默认), All-Trigger, Global γ=0.01, Global γ=0.05

In [ ]:
trigger_strategies = {
    "Base": None,
    "All": "trigger_all",
    "G-0.01": "trigger_global_gamma_global_0.01",
    "G-0.05": "trigger_global_gamma_global_0.05",
}

args = {**common_args, "epochs": 1,
        "lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}
num_clients = args.get("num_clients", 10)
trigger_counts = {}

for label, ablate_name in trigger_strategies.items():
    data = cached_load("dfedset", ablate_name=ablate_name, **args, specific_run=0)
    if data is None:
        print(f"  [跳过] {label}: 无结果")
        continue

    triggered_ids = data["triggered_ids"]
    counts = [0] * num_clients
    for round_ids in triggered_ids:
        for cid in round_ids:
            if cid < num_clients:
                counts[cid] += 1
    trigger_counts[label] = counts

if trigger_counts:
    x = np.arange(num_clients)
    width = 0.2
    colors = ["#D62728", "#1F77B4", "#FF7F0E", "#2CA02C"]

    plt.figure(figsize=(10, 5))
    for i, (label, counts) in enumerate(trigger_counts.items()):
        plt.bar(x + i * width, counts, width, label=label, color=colors[i])

    plt.grid(False)
    plt.xlabel("Client ID", fontsize=plot_fontsize)
    plt.ylabel("Trigger Count", fontsize=plot_fontsize)
    plt.xticks(x + width * 1.5, [str(i) for i in range(num_clients)], fontsize=plot_fontsize)
    plt.legend(fontsize=plot_fontsize-4, loc='upper center', ncol=4)
    plt.yscale('log')
    plt.yticks(fontsize=plot_fontsize)
    plt.ylim(1, 3500)
    plt.tight_layout()
    plt.savefig("figures/trigger_strategies.pdf", bbox_inches="tight")
    plt.show()

## Fig 3: 超参数分析 — λ_sa × λ_so 热力图

**所需实验：** 5×5 = 25 组实验
遍历 λ_sa ∈ {1.0, 5.0, 10.0, 20.0, 50.0}, λ_so ∈ {0.01, 0.05, 0.1, 0.5, 1.0}
DFedSET, cifar10, epochs=1, alpha=0.1

In [ ]:
lambda_sa_list = [20]
lambda_so_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
model_acc_grid = np.full((len(lambda_sa_list), len(lambda_so_list)), np.nan)

for i, l_sa in enumerate(lambda_sa_list):
    for j, l_so in enumerate(lambda_so_list):
        args = {**common_args, "epochs": 1, "num_clients": 20,
                "lambda_sa": l_sa, "eta": 0.9, "lambda_so": l_so}
        seed_maxes = []
        for s in range(5):
            data = cached_load("dfedset", **args, specific_run=s)
            if data is not None and "acc" in data:
                seed_maxes.append(max(data["acc"]["model"]))
        if seed_maxes:
            model_acc_grid[i, j] = np.mean(seed_maxes)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(model_acc_grid, cmap="viridis", aspect="auto")
ax.grid(False)
ax.set_xticks(range(len(lambda_so_list)))
ax.set_xticklabels(lambda_so_list, fontsize=plot_fontsize)
ax.set_yticks(range(len(lambda_sa_list)))
ax.set_yticklabels(lambda_sa_list, fontsize=plot_fontsize)
ax.set_xlabel(r"$\lambda_{so}$", fontsize=plot_fontsize)
ax.set_ylabel(r"$\lambda_{sa}$", fontsize=plot_fontsize)
ax.set_title("Model Accuracy", fontsize=plot_fontsize)
cbar = plt.colorbar(im, ax=ax, fraction=0.046)
cbar.ax.tick_params(labelsize=plot_fontsize)
for ii in range(len(lambda_sa_list)):
    for jj in range(len(lambda_so_list)):
        val = model_acc_grid[ii, jj]
        if not np.isnan(val):
            color = "w" if val < np.nanmax(model_acc_grid) * 0.7 else "k"
            ax.text(jj, ii, f"{val:.2f}", ha="center", va="center", color=color, fontsize=plot_fontsize-2)

plt.tight_layout()
plt.savefig("figures/hyperparam_heatmap.pdf", bbox_inches="tight")
plt.show()

## Fig 4: 消融实验（4 个维度）

**所需实验（共 10 组，每个维度独立消融，其他保持默认）：**

| 维度 | 变量 | 取值 |
|---|---|---|
| A-Relay | `ablate.relay` | `true`(默认), `false` |
| B-Aggregator | `ablate.aggregator` | `true`(默认·MH redirect), `false`(plain avg) |
| C-Confidence | `ablate.confidence` | `"log"`(默认), `"count"`, `"none"` |
| D-Trigger | `ablate.trigger` | `"adaptive"`(默认), `"global"`, `"all"` |

固定 λ_sa=10.0, λ_so=0.1, epochs=1

In [ ]:
# Ablation experiment config: ablate_name maps to subdirectory name

ablation_configs = {
    "Base":                    {"ablate_name": None, "category": "DFedSET"},
    "-":                       {"ablate_name": "relay", "category": "Relay"},
    "Plain Aggr.":             {"ablate_name": "aggregator", "category": "Aggregation"},
    "count":                   {"ablate_name": "confidence_count", "category": "Weighting"},
    "none":                    {"ablate_name": "confidence_none", "category": "Weighting"},
    "global (0.01)":           {"ablate_name": "trigger_global_gamma_global_0.01", "category": "Trigger"},
    "global (0.05)":           {"ablate_name": "trigger_global_gamma_global_0.05", "category": "Trigger"},
    "all":                     {"ablate_name": "trigger_all", "category": "Trigger"},
}

ablation_results = {}

base_kwargs = {**common_args, "lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}
NUM_SEEDS = 5

for label, cfg in ablation_configs.items():
    seed_maxes = []
    seed_tr = []

    for s in range(NUM_SEEDS):
        data = cached_load("dfedset", ablate_name=cfg["ablate_name"],
                           **base_kwargs, specific_run=s)
        if data is None:
            continue
        seed_maxes.append(max(data["acc"]["model"]))
        if "num_triggered" in data:
            tr = np.mean(data["num_triggered"]) / 10.0 * 100
        else:
            tr = 100.0
        seed_tr.append(tr)

    if not seed_maxes:
        continue

    m_max = np.mean(seed_maxes)
    trig_rate = np.mean(seed_tr)

    ablation_results[label] = {
        "category": cfg["category"],
        "model_max": m_max,
        "trig_rate": trig_rate
    }

# Build DataFrame
df_data = []
for label, res in ablation_results.items():
    df_data.append({
        "Component": res["category"],
        "Configuration": label,
        "Acc": f"{res['model_max']:.2f}",
        "Trigger Rate": f"{res['trig_rate']:.2f}"
    })
df = pd.DataFrame(df_data)
display(Markdown("### Ablation Study Results"))
display(df)

print("\nLaTeX format:")
print("\\begin{table}[htbp]")
print("\\centering")
print(df.to_latex(index=False, column_format="cccc"))
print("\\end{table}")

## Fig 5: GSD vs 触发数 vs Round

**所需实验：** 仅需一次完整的 DFedSET run（默认配置，1000 rounds）

In [ ]:
args = {**common_args, "epochs": 1,
        "lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}
data = cached_load("dfedset", **args, specific_run=0)
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
gsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"]["model"][:x_lim])     # (R,)

In [ ]:
# 前面的数据读取和处理保持原样
args = {**common_args, "epochs": 1,
        "lambda_sa": 20.0, "eta": 0.9, "lambda_so": 0.5}
data = cached_load("dfedset", **args, specific_run=0)
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
gsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"]["model"][:x_lim])      # (R,)

# --- 主图：双子图 (使用 sharex=True 保证对齐) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

color1, color2, color3 = "#4C72B0", "#DD8452", "#55A868"
mean_gsd = gsd.mean(axis=1)
min_gsd = gsd.min(axis=1)
max_gsd = gsd.max(axis=1)

# 计算自适应阈值 (EMA)
eta = 0.9
ema = np.zeros_like(gsd)
ema[0] = 0.5
if x_lim > 1:
    ema[1] = gsd[1]
for r in range(2, x_lim):
    ema[r] = eta * ema[r-1] + (1.0 - eta) * gsd[r]
mean_ema = ema.mean(axis=1)

# --- 上子图：GSD 与自适应阈值 ---
ax1.plot(mean_gsd, color=color1, label="Mean GSD", linewidth=plot_linewidth)
ax1.fill_between(range(x_lim),
    min_gsd, max_gsd,
    alpha=0.15, color=color1)
ax1.plot(mean_ema, color="#E24A33", linestyle="--", label="Mean Threshold", linewidth=plot_linewidth + 0.5)
ax1.set_ylabel("GSD", color="black", fontsize=plot_fontsize)

# 强制 ax1 的底部严格为 0，使曲线基于坐标轴基线
ax1.set_ylim(0, 0.2)
ax1.set_yticks([0.1, 0.2])
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelsize=plot_fontsize)
# ax1.axvspan(0, 2, alpha=0.1, color="gray", label="Warmup")
ax1.legend(loc="upper right", fontsize=plot_fontsize-2)

# --- 下子图：触发客户端数 ---
ax2.bar(range(x_lim), num_tr, color=color2, alpha=0.6, width=0.8)
ax2.set_ylabel("# Triggered Clients", color="black", fontsize=plot_fontsize)

# 强制 ax2 的顶部严格为 0
ax2.set_ylim(10.5, 0)
ax2.set_yticks([0, 2, 4, 6, 8, 10])
ax2.grid(True, alpha=0.3)

# 采用方案4：数值与标签全部放在最底部
ax2.tick_params(labelsize=plot_fontsize)
ax2.set_xlabel("Round", fontsize=plot_fontsize)
ax2.xaxis.set_label_position('bottom')
ax2.set_xlim(0, x_lim)

# 将上下子图严丝合缝拼在一起
fig.subplots_adjust(hspace=0)
fig.savefig("figures/gsd_trigger.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 数据读取：全局阈值消融 (gamma=0.01)
args = {**common_args, "epochs": 1,
        "lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}
data = cached_load("dfedset", **args, specific_run=0)
start_round = 5  # 避开前几轮
# 构造触发状态矩阵 (R, C)
trigger_matrix = np.zeros_like(gsd)
for r in range(x_lim):
    for cid in data["triggered_ids"][r]:
        if cid < gsd.shape[1]:
            trigger_matrix[r, cid] = 1.0

fig, (ax_h1, ax_h2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# GSD 热力图 (使用 YlOrRd，从下往上为 0--9)
im1 = ax_h1.imshow(gsd[start_round:x_lim].T[::-1, :], aspect="auto", cmap="YlOrRd",
                   extent=[start_round, x_lim, 0, 9])
ax_h1.set_yticks([2, 4, 6, 8])
ax_h1.tick_params(labelsize=plot_fontsize)
ax_h1.grid(False)  # 禁用热力图网格线
cbar1 = plt.colorbar(im1, ax=ax_h1, label="GSD", fraction=0.046, pad=0.04)
cbar1.ax.yaxis.label.set_fontsize(plot_fontsize)
cbar1.ax.tick_params(labelsize=plot_fontsize)
ax_h1.text(-0.115, 0.8, "(a)", transform=ax_h1.transAxes, fontsize=plot_fontsize, fontweight="bold", va="bottom", ha="left")

# 触发状态热力图 (使用离散两色映射，从上往下为 0--9)
cmap_discrete = ListedColormap(["#DFF1F1", "#2C5EAD"])
norm_discrete = BoundaryNorm([0, 0.5, 1], cmap_discrete.N)

im2 = ax_h2.imshow(trigger_matrix[start_round:x_lim].T[::-1, :], aspect="auto", cmap=cmap_discrete, norm=norm_discrete,
                   extent=[start_round, x_lim, 0, 9])
ax_h2.set_xlabel("Round", fontsize=plot_fontsize)
ax_h2.set_yticks([2, 4, 6, 8])
ax_h2.tick_params(labelsize=plot_fontsize)
ax_h2.grid(False)  # 禁用热力图网格线

cbar2 = plt.colorbar(im2, ax=ax_h2, label="Triggered", fraction=0.046, pad=0.04, ticks=[0.25, 0.75])
cbar2.ax.yaxis.label.set_fontsize(plot_fontsize)
cbar2.ax.tick_params(labelsize=plot_fontsize)
cbar2.ax.set_yticklabels(["0", "1"])

fig.subplots_adjust(hspace=0)
fig.supylabel("Client ID", fontsize=plot_fontsize, x=0.05)
fig.savefig("figures/gsd_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 数据读取：全局阈值消融 (gamma=0.01)
args = {**common_args, "epochs": 1,
        "lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}
data = cached_load("dfedset", ablate_name="trigger_global_gamma_global_0.01",
                   **args, specific_run=0)
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
gsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"]["model"][:x_lim])      # (R,)

start_round = 5  # 避开前几轮
# 构造触发状态矩阵 (R, C)
trigger_matrix = np.zeros_like(gsd)
for r in range(x_lim):
    for cid in data["triggered_ids"][r]:
        if cid < gsd.shape[1]:
            trigger_matrix[r, cid] = 1.0

fig, (ax_h1, ax_h2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# GSD 热力图 (使用 YlOrRd，从下往上为 0--9)
im1 = ax_h1.imshow(gsd[start_round:x_lim].T[::-1, :], aspect="auto", cmap="YlOrRd",
                   extent=[start_round, x_lim, 0, 9])
ax_h1.set_yticks([2, 4, 6, 8])
ax_h1.tick_params(labelsize=plot_fontsize)
ax_h1.grid(False)  # 禁用热力图网格线
cbar1 = plt.colorbar(im1, ax=ax_h1, label="GSD", fraction=0.046, pad=0.04)
cbar1.ax.yaxis.label.set_fontsize(plot_fontsize)
cbar1.ax.tick_params(labelsize=plot_fontsize)
ax_h1.text(-0.115, 0.8, "(b)", transform=ax_h1.transAxes, fontsize=plot_fontsize, fontweight="bold", va="bottom", ha="left")

# 触发状态热力图 (使用离散两色映射，从上往下为 0--9)
cmap_discrete = ListedColormap(["#DFF1F1", "#2C5EAD"])
norm_discrete = BoundaryNorm([0, 0.5, 1], cmap_discrete.N)

im2 = ax_h2.imshow(trigger_matrix[start_round:x_lim].T[::-1, :], aspect="auto", cmap=cmap_discrete, norm=norm_discrete,
                   extent=[start_round, x_lim, 0, 9])
ax_h2.set_xlabel("Round", fontsize=plot_fontsize)
ax_h2.set_yticks([2, 4, 6, 8])
ax_h2.tick_params(labelsize=plot_fontsize)
ax_h2.grid(False)  # 禁用热力图网格线

cbar2 = plt.colorbar(im2, ax=ax_h2, label="Triggered", fraction=0.046, pad=0.04, ticks=[0.25, 0.75])
cbar2.ax.yaxis.label.set_fontsize(plot_fontsize)
cbar2.ax.tick_params(labelsize=plot_fontsize)
cbar2.ax.set_yticklabels(["0", "1"])

fig.subplots_adjust(hspace=0)
fig.supylabel("Client ID", fontsize=plot_fontsize, x=0.05)
fig.savefig("figures/gsd_heatmap_global_0.01.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 数据读取：全局阈值消融 (gamma=0.05)
args = {**common_args, "epochs": 1,
        "lambda_sa": 10.0, "eta": 0.9, "lambda_so": 0.1}
data = cached_load("dfedset", ablate_name="trigger_global_gamma_global_0.05",
                   **args, specific_run=0)
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
gsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"]["model"][:x_lim])      # (R,)

start_round = 5  # 避开前几轮
# 构造触发状态矩阵 (R, C)
trigger_matrix = np.zeros_like(gsd)
for r in range(x_lim):
    for cid in data["triggered_ids"][r]:
        if cid < gsd.shape[1]:
            trigger_matrix[r, cid] = 1.0

fig, (ax_h1, ax_h2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# GSD 热力图 (使用 YlOrRd，从下往上为 0--9)
im1 = ax_h1.imshow(gsd[start_round:x_lim].T[::-1, :], aspect="auto", cmap="YlOrRd",
                   extent=[start_round, x_lim, 0, 9])
ax_h1.set_yticks([2, 4, 6, 8])
ax_h1.tick_params(labelsize=plot_fontsize)
ax_h1.grid(False)  # 禁用热力图网格线
cbar1 = plt.colorbar(im1, ax=ax_h1, label="GSD", fraction=0.046, pad=0.04)
cbar1.ax.yaxis.label.set_fontsize(plot_fontsize)
cbar1.ax.tick_params(labelsize=plot_fontsize)
ax_h1.text(-0.115, 0.8, "(c)", transform=ax_h1.transAxes, fontsize=plot_fontsize, fontweight="bold", va="bottom", ha="left")

# 触发状态热力图 (使用离散两色映射，从上往下为 0--9)
cmap_discrete = ListedColormap(["#DFF1F1", "#2C5EAD"])
norm_discrete = BoundaryNorm([0, 0.5, 1], cmap_discrete.N)

im2 = ax_h2.imshow(trigger_matrix[start_round:x_lim].T[::-1, :], aspect="auto", cmap=cmap_discrete, norm=norm_discrete,
                   extent=[start_round, x_lim, 0, 9])
ax_h2.set_xlabel("Round", fontsize=plot_fontsize)
ax_h2.set_yticks([2, 4, 6, 8])
ax_h2.tick_params(labelsize=plot_fontsize)
ax_h2.grid(False)  # 禁用热力图网格线

cbar2 = plt.colorbar(im2, ax=ax_h2, label="Triggered", fraction=0.046, pad=0.04, ticks=[0.25, 0.75])
cbar2.ax.yaxis.label.set_fontsize(plot_fontsize)
cbar2.ax.tick_params(labelsize=plot_fontsize)
cbar2.ax.set_yticklabels(["0", "1"])

fig.subplots_adjust(hspace=0)
fig.supylabel("Client ID", fontsize=plot_fontsize, x=0.05)
fig.savefig("figures/gsd_heatmap_global_0.05.pdf", bbox_inches="tight")
plt.show()